# Optical map: legacy CNP → MFGP

One notebook for the integrated offline RESuM pipeline. It uses the prepared LF
**non-zero-hit voxels**, trains the **legacy two-output CNP** with standard
class-aware mixup and **mixed context**, and fits the three-fidelity MFGP.

**Run All trains a fresh model by default.** Each run receives a new output
folder. Set `RUN_TRAINING = False` to inspect the existing completed run instead.
Use the repository `.venv` as the notebook kernel, with the GP and optical-data
dependencies installed. Run from the repository root or its `notebooks/` folder.

The default loss is **`theory-truth` Bernoulli NLL**, matching the optical run
configuration. To use the literal original Gaussian NLL, set
`LOSS_OVERRIDE = "practice-truth"`. Both retain the two-output architecture.
This notebook calls shared modules; it does not implement a separate training loop.

In [ ]:
import os
import sys
from pathlib import Path

for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "1"

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "config.optical.resum.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the RESUM_FLEX repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
from datetime import datetime, timezone
import numpy as np
import torch
from IPython.display import Image, Markdown, display
from schemas.surrogates import SurrogateRunConfig, load_surrogate_config
from core.surrogates import load_surrogate
from core.surrogates.experiment import run_experiment
from core.surrogate_mfgp import load_mfgp
from data.optical_pipeline import load_prepared_batch

torch.set_num_threads(1)
print("Repository:", ROOT)
print("Python:", sys.executable)

## Run settings

The YAML is the source of architecture, optimizer, sampling, and GP settings.
The overrides below affect a new run only. When viewing an existing run, the
notebook reports its saved configuration rather than relabeling it with overrides.

Standard class-aware mixup is the **single-branch objective** here: no extra
real-data loss, class weights, or positive-quota correction. The context/target
source split is fresh in each batch; both parents stay within their assigned side.

In [ ]:
RUN_TRAINING = True
LOSS_OVERRIDE = None  # None: YAML default; or "theory-truth" / "practice-truth"
EXISTING_RUN = None  # Or an explicit Path to a completed run

base = load_surrogate_config(ROOT / "config.optical.resum.yaml")
settings = base.model_dump(mode="python")
settings["model"]["kind"] = "legacy_cnp"
settings["training"]["objective"] = {"strategy": "single"}
settings["training"]["sampling"] = {
    "strategy": "class_aware_mixup",
    "mixup": {"alpha": 0.2, "mix_context": True},
}
settings["training"]["weighting"] = {"strategy": "none"}
settings["training"]["focal_gamma"] = 0
if LOSS_OVERRIDE is not None:
    settings["training"]["loss"] = LOSS_OVERRIDE
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
settings["output_directory"] = ROOT / "outputs/optical_resum/notebook" / run_id
config = SurrogateRunConfig.model_validate(settings)
assert config.mfgp is not None, "The full pipeline requires an mfgp configuration."

if RUN_TRAINING:
    print(json.dumps(config.model_dump(mode="json"), indent=2))
else:
    if EXISTING_RUN is None:
        completed = sorted((ROOT / "outputs/optical_resum/notebook").glob("*/mfgp/model.pkl"))
        EXISTING_RUN = completed[-1].parent.parent if completed else base.output_directory
    else:
        EXISTING_RUN = Path(EXISTING_RUN).expanduser()
        if not EXISTING_RUN.is_absolute():
            EXISTING_RUN = ROOT / EXISTING_RUN
    print("Viewing saved results:", EXISTING_RUN)

## Check the prepared data

Preparation is separate: this notebook uses saved normalized batches and split
metadata. It does not filter individual negative events or rebuild train/validation
splits. LF voxels must have at least one observed hit; HF zero-hit voxels are kept.
The CNP trains on LF only. The GP uses LF and HF training voxels. Test files are
never loaded. Excluding zero-hit LF voxels limits what that validation set represents.

In [ ]:
def show_table(rows, columns):
    def cell(value):
        return f"{value:.6g}" if isinstance(value, float) else str(value)
    text = "| " + " | ".join(columns) + " |\n"
    text += "| " + " | ".join(["---"] * len(columns)) + " |\n"
    text += "\n".join("| " + " | ".join(cell(row.get(c, "—")) for c in columns) + " |"
                      for row in rows)
    display(Markdown(text))

if RUN_TRAINING:
    counts = []
    for split in ("train", "validation"):
        for fidelity in ("lf", "hf"):
            path = config.data_directory / "batches" / split / f"{fidelity}.npz"
            if not path.exists():
                raise FileNotFoundError(f"Missing prepared input: {path}")
            batch = load_prepared_batch(path)
            zero = int((batch.labels.sum(axis=1) == 0).sum())
            if fidelity == "lf" and zero:
                raise ValueError("Expected LF non-zero-hit voxels; check data_directory.")
            counts.append(dict(split=split, fidelity=fidelity, voxels=batch.batch_size,
                               events_per_voxel=batch.n_events, hits=int(batch.labels.sum()),
                               zero_hit_voxels=zero))
    show_table(counts, ["split", "fidelity", "voxels", "events_per_voxel", "hits", "zero_hit_voxels"])
else:
    if not (EXISTING_RUN / "mfgp/model.pkl").exists():
        raise FileNotFoundError("No complete saved pipeline at EXISTING_RUN; train or select another run.")

## Train the CNP and fit the MFGP

`run_experiment` executes the full pipeline:

1. Train the CNP from scratch and select the checkpoint by LF validation MAE.
2. Evaluate real LF/HF events and save CNP diagnostics and checkpoints.
3. Construct GP training levels: LF CNP means, HF CNP means, HF raw fractions.
4. Fit the GP and save predictions, normalization, plots, and the development map.

This is a synchronous run in the notebook kernel, on the configured device.
It does not submit a Slurm job. The prior CPU run took about four minutes;
runtime depends on the machine. The cell may remain quiet during fitting.

In [ ]:
if RUN_TRAINING:
    from time import perf_counter
    started = perf_counter()
    print("Starting full pipeline:", config.output_directory, flush=True)
    RUN_DIR = run_experiment(config)
    print(f"Completed in {perf_counter() - started:.1f} seconds.")
else:
    RUN_DIR = EXISTING_RUN

manifest = json.loads((RUN_DIR / "experiment.json").read_text())
actual_config = manifest["resolved_config"]
best = json.loads((RUN_DIR / "checkpoints/best/model.json").read_text())
print("Results:", RUN_DIR)
print("Selected CNP step:", best["metadata"]["step"])
print("Actual loss:", actual_config["training"]["loss"])
print("Actual mixup:", actual_config["training"]["sampling"])
print("Actual objective:", actual_config["training"]["objective"])

## CNP results on real events

Selection uses LF validation only. HF metrics are held-out reporting; training PR
is in-sample. The mean/residual plots show the selected and final checkpoints.

In [ ]:
cnp_metrics = json.loads((RUN_DIR / "metrics.json").read_text())
show_table([r for r in cnp_metrics if r["checkpoint"] == "best" and r["split"] == "validation"],
           ["fidelity", "mean_observed", "mean_predicted", "voxel_rate_mae", "pearson_r", "average_precision"])
print("Training sampling audit:")
display(json.loads((RUN_DIR / "sampling_audit.json").read_text()))

def show_plot(path):
    if not path.is_file():
        raise FileNotFoundError(f"Expected pipeline plot: {path}")
    display(Image(filename=str(path)))

for name in ("validation_history", "lf_means", "hf_means", "lf_precision_recall", "hf_precision_recall"):
    show_plot(RUN_DIR / f"{name}.png")

## MFGP results and ±1σ/±2σ/±3σ coverage

The **HF highest-level output** estimates the simulation detection fraction.
The LF level is fitted to CNP means; comparing it with raw LF fractions also
reveals residual CNP bias. These are distinct diagnostics.

Coverage plots use the existing `viz.dispatch.plot_coverage_test` through the
shared pipeline. Bands include the GP's fitted observation noise; they are not
CNP uncertainty or pure latent-function bands. The Gaussian GP is unconstrained,
so negative predictions/bands are displayed without clipping. Ten HF validation
voxels provide only a preliminary coverage assessment.

In [ ]:
GP_DIR = RUN_DIR / "mfgp"
gp_metrics = json.loads((GP_DIR / "metrics.json").read_text())
show_table(gp_metrics, ["fidelity", "level", "voxels", "mean_observed", "mean_predicted", "mae", "pearson_r"])
show_table([dict(fidelity=r["fidelity"], **{f"{k}σ": r["coverage"][str(k)] for k in (1, 2, 3)})
            for r in gp_metrics], ["fidelity", "1σ", "2σ", "3σ"])
for fidelity in ("hf", "lf"):
    show_plot(GP_DIR / f"{fidelity}_gp_means.png")
    show_plot(GP_DIR / f"{fidelity}_coverage.png")

## Optical map and loading the trained models

The saved map evaluates the highest-fidelity GP at available **development
coordinates**, not a dense spatial grid and not test coordinates. New queries
must use the original theta normalization. Outputs remain detection fractions.

In [ ]:
show_plot(GP_DIR / "development_map.png")
cnp = load_surrogate(RUN_DIR / "checkpoints/best")
mfgp = load_mfgp(GP_DIR / "model.pkl")
normalization = json.loads((GP_DIR / "normalization.json").read_text())["theta"]
with np.load(GP_DIR / "development_map.npz") as saved:
    physical_positions = saved["theta_physical"][:3].copy()
    reference_means = saved["mean"][:3].copy()
normalized_positions = (physical_positions - np.asarray(normalization["offset"])) / np.asarray(normalization["scale"])
mean, variance = mfgp.predict(normalized_positions, fidelity=2)
np.testing.assert_allclose(mean, reference_means, rtol=1e-6, atol=1e-10)
show_table([dict(x=float(x[0]), y=float(x[1]), z=float(x[2]),
                 detection_fraction=float(m), observation_sigma=float(np.sqrt(v)))
            for x, m, v in zip(physical_positions, mean, variance, strict=True)],
           ["x", "y", "z", "detection_fraction", "observation_sigma"])
print("CNP checkpoint:", RUN_DIR / "checkpoints/best")
print("MFGP checkpoint:", GP_DIR / "model.pkl")
print("All metrics, figures and run settings:", RUN_DIR)